In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

In [ ]:
samples = 500

engine_size = np.random.uniform(1.0, 5.0, samples)
horsepower = 65 + engine_size * 42 + np.random.normal(0, 15, samples)
vehicle_age = np.random.uniform(0, 15, samples)

price = (
    25000
    + horsepower * 75
    - vehicle_age * 1350
    + np.random.normal(0, 1400, samples)
)

vehicle_df = pd.DataFrame({
    "engine_size_L": engine_size,
    "horsepower": horsepower,
    "age_years": vehicle_age,
    "price": price
})

vehicle_df.head()

,engine_size_L,horsepower,age_years,price
0,2.498160,175.049080,4.005424,32990.227262
1,4.802857,294.862566,13.179450,28889.672181
2,3.927976,244.231340,11.961390,27356.430814
3,3.394634,198.921071,9.876778,26371.972865
4,1.624075,119.734912,12.758726,17747.190498


In [ ]:
!pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("vehicle_price_pipeline")

X = vehicle_df.drop(columns=["price"])
y = vehicle_df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )
}

run_info = {}

for model_name, model in models.items():

    with mlflow.start_run(run_name=model_name) as run:

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)

        mlflow.log_param("model_type", model_name)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)

        mlflow.sklearn.log_model(
            sk_model=model,
            name="vehicle_model",
            serialization_format="pickle"
        )

        run_info[model_name] = run.info.run_id

        print(model_name)
        print("Run ID :", run.info.run_id)
        print("MAE    :", round(mae, 2))
        print("RMSE   :", round(rmse, 2))
        print("R2     :", round(r2, 3))
        print("-" * 40)

2026/09/24 14:02:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear Regression
Run ID : 28d1672ad05e4243a0766c907f647e1e
MAE    : 1339.85
RMSE   : 1650.51
R2     : 0.949
----------------------------------------


2026/09/24 14:02:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest
Run ID : 0d34292ea97448449fb518be24aa1a5e
MAE    : 1480.12
RMSE   : 1901.44
R2     : 0.933
----------------------------------------


In [ ]:
client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name("vehicle_price_pipeline")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.RMSE ASC"]
)

best_run = runs[0]

print("Best Model:", best_run.data.params["model_type"])
print("Best RMSE :", round(best_run.data.metrics["RMSE"],2))

registered_model = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/vehicle_model",
    name="vehicle_price_predictor"
)

print("Registered Model Version:", registered_model.version)

Successfully registered model 'vehicle_price_predictor'.
2026/09/24 14:02:41 WARNING mlflow.tracking._model_registry.fluent: Run with id 28d1672ad05e4243a0766c907f647e1e has no artifacts at artifact path 'vehicle_model', registering model based on models:/m-57659549d67c4fdc8233480848559eec instead


Best Model: Linear Regression
Best RMSE : 1650.51
Registered Model Version: 1


Created version '1' of model 'vehicle_price_predictor'.


In [ ]:
dockerfile = f"""
FROM python:3.10-slim

WORKDIR /app

RUN pip install mlflow scikit-learn pandas numpy

COPY mlruns /app/mlruns

ENV MLFLOW_TRACKING_URI=file:/app/mlruns

EXPOSE 5001

CMD ["mlflow","models","serve",
     "-m","models:/vehicle_price_predictor/{registered_model.version}",
     "-h","0.0.0.0",
     "-p","5001"]
"""

with open("Dockerfile", "w") as file:
    file.write(dockerfile)

print(dockerfile)


FROM python:3.10-slim

WORKDIR /app

RUN pip install mlflow scikit-learn pandas numpy

COPY mlruns /app/mlruns

ENV MLFLOW_TRACKING_URI=file:/app/mlruns

EXPOSE 5001

CMD ["mlflow","models","serve",
     "-m","models:/vehicle_price_predictor/1",
     "-h","0.0.0.0",
     "-p","5001"]



In [ ]:
github_actions = """
name: vehicle-mlops-pipeline

on:
  push:
    branches: [main]

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install Libraries
        run: |
          pip install mlflow scikit-learn pandas numpy

      - name: Train ML Model
        run: python train_model.py

      - name: Build Docker Image
        run: docker build -t vehicle-price-model .

      - name: Deploy Model
        run: |
          docker run -d -p 5001:5001 vehicle-price-model
"""

with open("github_actions.yml", "w") as file:
    file.write(github_actions)

print(github_actions)


name: vehicle-mlops-pipeline

on:
  push:
    branches: [main]

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install Libraries
        run: |
          pip install mlflow scikit-learn pandas numpy

      - name: Train ML Model
        run: python train_model.py

      - name: Build Docker Image
        run: docker build -t vehicle-price-model .

      - name: Deploy Model
        run: |
          docker run -d -p 5001:5001 vehicle-price-model



In [ ]:
print("MLOps Workflow Simulation Completed Successfully.\n")

print("Generated Files:")
print("1. MLflow Tracking Database  -> mlflow.db")
print("2. Docker Deployment File    -> Dockerfile")
print("3. GitHub Actions Pipeline   -> github_actions.yml")

print("\nRegistered Model Name : vehicle_price_predictor")
print("Registered Version    :", registered_model.version)

MLOps Workflow Simulation Completed Successfully.

Generated Files:
1. MLflow Tracking Database  -> mlflow.db
2. Docker Deployment File    -> Dockerfile
3. GitHub Actions Pipeline   -> github_actions.yml

Registered Model Name : vehicle_price_predictor
Registered Version    : 1
